# CNN on CIFAR-10 (with MNIST Fallback)

---

## Learning Objectives

By the end of this notebook, you will:

- Build a **complete CNN architecture** from scratch: Conv -> ReLU -> Pool -> FC
- Load and preprocess **CIFAR-10** (or MNIST as fallback) with proper data pipelines
- Train a CNN **end-to-end** with training/validation splits
- Track and **plot loss and accuracy curves** over epochs
- Evaluate the model with a **confusion matrix**
- Understand **model capacity** and the effect of depth

## Prerequisites

- **DL150**: PyTorch Foundations (tensors, autograd, nn.Module)
- **DL200**: ANN/MLP Practical (training loops, DataLoaders)
- **Notebook 01**: CNN Basics (convolutions, pooling, output size formula)

## Table of Contents

1. [Setup & Data Loading](#1.-Setup-&-Data-Loading)
2. [Exploring the Dataset](#2.-Exploring-the-Dataset)
3. [Building the CNN Architecture](#3.-Building-the-CNN-Architecture)
4. [Training Pipeline](#4.-Training-Pipeline)
5. [Training the Model](#5.-Training-the-Model)
6. [Evaluation & Visualization](#6.-Evaluation-&-Visualization)
7. [Model Capacity and Depth](#7.-Model-Capacity-and-Depth)
8. [Common Mistakes & Debugging Tips](#8.-Common-Mistakes-&-Debugging-Tips)
9. [Exercises](#9.-Exercises)
10. [Exercise Solutions](#10.-Exercise-Solutions)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from src.utils.device import get_device
from src.utils.plotting import plot_loss_curves, plot_confusion_matrix
from src.utils.metrics import compute_accuracy

device = get_device()
print(f"Device: {device}")

---

## 1. Setup & Data Loading

We will use **CIFAR-10** (10 classes, 32x32 RGB images). If the download fails (e.g., no internet), we fall back to **MNIST** (10 classes, 28x28 grayscale).

**Data pipeline:**
- Convert images to tensors
- Normalize to zero mean and unit variance (per channel)
- Split training set into train/validation (90/10)

In [ ]:
# --- Data Loading with CIFAR-10 / MNIST fallback ---
BATCH_SIZE = 64
USE_CIFAR = True  # Will be set to False if CIFAR download fails

# CIFAR-10 normalization values (precomputed from training set)
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

# MNIST normalization
MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)

try:
    # Try CIFAR-10
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    
    full_train_dataset = datasets.CIFAR10(
        root='../../data', train=True, download=True, transform=train_transform
    )
    test_dataset = datasets.CIFAR10(
        root='../../data', train=False, download=True, transform=test_transform
    )
    
    CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    IN_CHANNELS = 3
    IMG_SIZE = 32
    print("Successfully loaded CIFAR-10")
    
except Exception as e:
    print(f"CIFAR-10 download failed ({e}). Falling back to MNIST.")
    USE_CIFAR = False
    
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD),
    ])
    
    full_train_dataset = datasets.MNIST(
        root='../../data', train=True, download=True, transform=train_transform
    )
    test_dataset = datasets.MNIST(
        root='../../data', train=False, download=True, transform=test_transform
    )
    
    CLASS_NAMES = [str(i) for i in range(10)]
    IN_CHANNELS = 1
    IMG_SIZE = 28
    print("Successfully loaded MNIST")

# Split training into train/val (90/10)
set_global_seed(42)
n_train = int(0.9 * len(full_train_dataset))
n_val = len(full_train_dataset) - n_train
train_dataset, val_dataset = random_split(full_train_dataset, [n_train, n_val])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nDataset: {'CIFAR-10' if USE_CIFAR else 'MNIST'}")
print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")
print(f"Image shape:        ({IN_CHANNELS}, {IMG_SIZE}, {IMG_SIZE})")
print(f"Number of classes:  {len(CLASS_NAMES)}")

---

## 2. Exploring the Dataset

Always visualize your data before training. Check:
- Are the images what you expect?
- Are labels correct?
- What does the class distribution look like?

In [ ]:
# Show sample images with their labels
def show_samples(dataset, class_names, n_samples=16, title="Sample Images"):
    """Display a grid of sample images from the dataset."""
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    
    for idx, ax in zip(indices, axes.flat):
        image, label = dataset[idx]
        
        # Unnormalize for display
        if image.shape[0] == 3:  # RGB
            mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
            std = torch.tensor(CIFAR_STD).view(3, 1, 1)
            img_display = (image * std + mean).clamp(0, 1)
            img_display = img_display.permute(1, 2, 0).numpy()  # CHW -> HWC
        else:  # Grayscale
            mean = torch.tensor(MNIST_MEAN).view(1, 1, 1)
            std = torch.tensor(MNIST_STD).view(1, 1, 1)
            img_display = (image * std + mean).clamp(0, 1)
            img_display = img_display.squeeze(0).numpy()  # Remove channel dim
        
        ax.imshow(img_display, cmap='gray' if image.shape[0] == 1 else None)
        ax.set_title(class_names[label], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


np.random.seed(42)
show_samples(full_train_dataset, CLASS_NAMES, title=f"Sample Images from {'CIFAR-10' if USE_CIFAR else 'MNIST'}")

In [ ]:
# Class distribution
if hasattr(full_train_dataset, 'targets'):
    all_labels = np.array(full_train_dataset.targets)
else:
    all_labels = np.array([full_train_dataset[i][1] for i in range(len(full_train_dataset))])

unique, counts = np.unique(all_labels, return_counts=True)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASS_NAMES, counts, color='steelblue', edgecolor='white')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('Class Distribution (Training Set)')
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---

## 3. Building the CNN Architecture

We build a classic CNN with the following structure:

```
Conv(3x3, 32) -> ReLU -> MaxPool(2x2)
Conv(3x3, 64) -> ReLU -> MaxPool(2x2)
Flatten
FC(hidden) -> ReLU -> Dropout
FC(10)  # 10 classes
```

**Dimension tracking (CIFAR-10: 3x32x32):**
1. Input: `(B, 3, 32, 32)`
2. Conv1 (3x3, pad=1): `(B, 32, 32, 32)` -> Pool: `(B, 32, 16, 16)`
3. Conv2 (3x3, pad=1): `(B, 64, 16, 16)` -> Pool: `(B, 64, 8, 8)`
4. Flatten: `(B, 64*8*8)` = `(B, 4096)`
5. FC1: `(B, 128)` -> FC2: `(B, 10)`

In [ ]:
class SimpleCNN(nn.Module):
    """A simple 2-layer CNN for image classification.
    
    Architecture: Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> Flatten -> FC -> FC
    """
    
    def __init__(self, in_channels=3, img_size=32, num_classes=10):
        super().__init__()
        
        # Convolutional feature extractor
        self.features = nn.Sequential(
            # Block 1: Conv -> ReLU -> MaxPool
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),  # -> (32, H, W)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # -> (32, H/2, W/2)
            
            # Block 2: Conv -> ReLU -> MaxPool
            nn.Conv2d(32, 64, kernel_size=3, padding=1),           # -> (64, H/2, W/2)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                 # -> (64, H/4, W/4)
        )
        
        # Compute the flattened size dynamically
        # This avoids hard-coding and works for any image size
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, img_size, img_size)
            flat_size = self.features(dummy).view(1, -1).shape[1]
        
        # Fully connected classifier
        self.classifier = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten: (B, C, H, W) -> (B, C*H*W)
        x = self.classifier(x)
        return x


# Instantiate model
model = SimpleCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE, num_classes=10).to(device)

# Model summary
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify with a dummy forward pass
dummy_input = torch.randn(2, IN_CHANNELS, IMG_SIZE, IMG_SIZE).to(device)
dummy_output = model(dummy_input)
print(f"\nDummy input shape:  {dummy_input.shape}")
print(f"Dummy output shape: {dummy_output.shape}")

---

## 4. Training Pipeline

We define a training loop that:
- Trains for a given number of epochs
- Tracks training loss, validation loss, training accuracy, and validation accuracy
- Prints progress every epoch

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, device):
    """Train a classification model and track metrics.
    
    Returns:
        history dict with keys: train_loss, val_loss, train_acc, val_acc
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }
    
    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = loss_fn(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
        
        train_loss = running_loss / total
        train_acc = correct / total
        
        # --- Validation ---
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = loss_fn(outputs, y_batch)
                
                val_running_loss += loss.item() * X_batch.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total
        
        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    return history

---

## 5. Training the Model

We train for 10 epochs with a learning rate of 0.001 (Adam optimizer).

In [ ]:
# Train the model
set_global_seed(42)
model = SimpleCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE, num_classes=10).to(device)

EPOCHS = 10
LR = 1e-3

history = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, device=device)

---

## 6. Evaluation & Visualization

Let us plot the training curves and evaluate on the test set.

In [ ]:
# Plot loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', markersize=4)
ax1.plot(epochs_range, history['val_loss'], 's-', label='Val Loss', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, history['train_acc'], 'o-', label='Train Acc', markersize=4)
ax2.plot(epochs_range, history['val_acc'], 's-', label='Val Acc', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('CNN Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nBest validation accuracy: {max(history['val_acc']):.4f} "
      f"(epoch {np.argmax(history['val_acc']) + 1})")

In [ ]:
# Evaluate on test set
test_acc = compute_accuracy(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Confusion matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

plot_confusion_matrix(all_labels, all_preds, class_names=CLASS_NAMES,
                      figsize=(8, 7), title=f"Confusion Matrix (Test Acc: {test_acc:.3f})")

In [ ]:
# Show some misclassified examples
misclassified_idx = np.where(all_preds != all_labels)[0]
n_show = min(10, len(misclassified_idx))
sample_idx = np.random.choice(misclassified_idx, n_show, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax_idx, data_idx in enumerate(sample_idx):
    image, label = test_dataset[data_idx]
    
    # Unnormalize for display
    if image.shape[0] == 3:
        mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
        std = torch.tensor(CIFAR_STD).view(3, 1, 1)
        img_display = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    else:
        mean = torch.tensor(MNIST_MEAN).view(1, 1, 1)
        std = torch.tensor(MNIST_STD).view(1, 1, 1)
        img_display = (image * std + mean).clamp(0, 1).squeeze(0).numpy()
    
    ax = axes[ax_idx // 5, ax_idx % 5]
    ax.imshow(img_display, cmap='gray' if image.shape[0] == 1 else None)
    ax.set_title(f"True: {CLASS_NAMES[all_labels[data_idx]]}\n"
                 f"Pred: {CLASS_NAMES[all_preds[data_idx]]}",
                 fontsize=9, color='red')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 7. Model Capacity and Depth

**Model capacity** refers to the range of functions a model can learn. Key factors:

- **Depth** (number of layers): Deeper networks can learn more abstract, hierarchical features
- **Width** (number of filters per layer): More filters capture more diverse features
- **Total parameters**: Rough proxy for capacity

**Trade-offs:**
- **Underfitting**: Model too small, cannot capture the data patterns (high train AND val error)
- **Overfitting**: Model too large, memorizes training data (low train error, high val error)
- **Sweet spot**: Good train/val performance with small gap

Let us compare a shallow vs deeper CNN:

In [ ]:
class DeeperCNN(nn.Module):
    """A deeper CNN with 3 conv blocks."""
    
    def __init__(self, in_channels=3, img_size=32, num_classes=10):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, img_size, img_size)
            flat_size = self.features(dummy).view(1, -1).shape[1]
        
        self.classifier = nn.Sequential(
            nn.Linear(flat_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# Compare parameter counts
shallow = SimpleCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE)
deep = DeeperCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE)

shallow_params = sum(p.numel() for p in shallow.parameters())
deep_params = sum(p.numel() for p in deep.parameters())

print(f"SimpleCNN (2 conv blocks): {shallow_params:>10,} parameters")
print(f"DeeperCNN (6 conv layers): {deep_params:>10,} parameters")
print(f"Ratio: {deep_params / shallow_params:.1f}x more parameters")

In [ ]:
# Train the deeper model for comparison (same number of epochs)
set_global_seed(42)
deep_model = DeeperCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE, num_classes=10).to(device)

print("Training DeeperCNN...")
deep_history = train_model(deep_model, train_loader, val_loader,
                           epochs=EPOCHS, lr=LR, device=device)

deep_test_acc = compute_accuracy(deep_model, test_loader, device)
print(f"\nDeeperCNN Test Accuracy: {deep_test_acc:.4f}")
print(f"SimpleCNN Test Accuracy: {test_acc:.4f}")

In [ ]:
# Compare training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss comparison
ax1.plot(epochs_range, history['val_loss'], 's-', label='SimpleCNN Val Loss', markersize=4)
ax1.plot(epochs_range, deep_history['val_loss'], 'o-', label='DeeperCNN Val Loss', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Validation Loss Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy comparison
ax2.plot(epochs_range, history['val_acc'], 's-', label='SimpleCNN Val Acc', markersize=4)
ax2.plot(epochs_range, deep_history['val_acc'], 'o-', label='DeeperCNN Val Acc', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('SimpleCNN vs DeeperCNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Common Mistakes & Debugging Tips

### Shape Mismatch at Flatten -> FC
- **Problem**: `RuntimeError: mat1 and mat2 shapes cannot be multiplied`
- **Fix**: Use the dummy-input trick to compute the flattened size:
  ```python
  dummy = torch.zeros(1, C, H, W)
  flat_size = self.features(dummy).view(1, -1).shape[1]
  ```

### Forgetting `model.eval()` and `torch.no_grad()`
- **Problem**: Validation loss fluctuates or dropout is still active during eval
- **Fix**: Always wrap validation/test in:
  ```python
  model.eval()
  with torch.no_grad():
      ...
  ```

### Not Normalizing Input Data
- **Problem**: Training is slow or unstable
- **Fix**: Always normalize with dataset-specific mean/std

### Learning Rate Too High
- **Symptom**: Loss oscillates or explodes
- **Fix**: Start with `lr=1e-3` for Adam, `lr=1e-2` for SGD with momentum

### Overfitting
- **Symptom**: Train accuracy >> Val accuracy, val loss increases
- **Fix**: Add dropout, data augmentation, weight decay, or reduce model size

### Channel Ordering
- PyTorch: **NCHW** (batch, channels, height, width)
- NumPy/PIL: **HWC** (height, width, channels)
- Use `.permute(1, 2, 0)` to convert CHW -> HWC for display

---

## 9. Exercises

### Exercise 1: Add More Conv Layers

Modify `SimpleCNN` to add a **third conv block** (Conv -> ReLU -> Pool with 128 filters). Train it and compare performance with the original.

### Exercise 2: Experiment with Hyperparameters

Try the following modifications and observe the effect:
- Change `kernel_size` from 3 to 5 in all conv layers
- Replace `MaxPool2d` with `AvgPool2d`
- Change the optimizer from Adam to SGD with momentum=0.9

### Exercise 3: Per-Class Accuracy

Compute and display accuracy for each individual class. Which classes are hardest?

In [ ]:
# Exercise space -- try the exercises here before looking at solutions

# Exercise 1: Three-block CNN
# YOUR CODE HERE


# Exercise 2: Hyperparameter experiments
# YOUR CODE HERE


# Exercise 3: Per-class accuracy
# YOUR CODE HERE


---

## 10. Exercise Solutions

In [ ]:
# --- Exercise 1 Solution: Three-block CNN ---
class ThreeBlockCNN(nn.Module):
    def __init__(self, in_channels=3, img_size=32, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3 (NEW)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, img_size, img_size)
            flat_size = self.features(dummy).view(1, -1).shape[1]
        
        self.classifier = nn.Sequential(
            nn.Linear(flat_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

set_global_seed(42)
three_block = ThreeBlockCNN(in_channels=IN_CHANNELS, img_size=IMG_SIZE).to(device)
params_3b = sum(p.numel() for p in three_block.parameters())
print(f"ThreeBlockCNN parameters: {params_3b:,}")
print(f"SimpleCNN parameters:     {shallow_params:,}")

# Train (using fewer epochs for speed in the solution demo)
print("\nTraining ThreeBlockCNN for 5 epochs...")
hist_3b = train_model(three_block, train_loader, val_loader,
                      epochs=5, lr=LR, device=device)
acc_3b = compute_accuracy(three_block, test_loader, device)
print(f"ThreeBlockCNN Test Accuracy: {acc_3b:.4f}")

In [ ]:
# --- Exercise 3 Solution: Per-class accuracy ---
print("Per-class accuracy on test set:")
print("-" * 35)

for class_idx, class_name in enumerate(CLASS_NAMES):
    mask = all_labels == class_idx
    class_correct = (all_preds[mask] == all_labels[mask]).sum()
    class_total = mask.sum()
    class_acc = class_correct / class_total if class_total > 0 else 0
    bar = '#' * int(class_acc * 30)
    print(f"  {class_name:<12s}: {class_acc:.4f} ({class_correct}/{class_total}) {bar}")

# Find hardest/easiest classes
per_class_acc = []
for i in range(len(CLASS_NAMES)):
    mask = all_labels == i
    per_class_acc.append((all_preds[mask] == all_labels[mask]).mean())

easiest = CLASS_NAMES[np.argmax(per_class_acc)]
hardest = CLASS_NAMES[np.argmin(per_class_acc)]
print(f"\nEasiest class: {easiest} ({max(per_class_acc):.4f})")
print(f"Hardest class: {hardest} ({min(per_class_acc):.4f})")